In [1]:
import os
import pandas as pd

In [2]:
MNES = {
    'barcelona': 'FAE',
    'cluj': 'CLJ'
}

SD_TYPES = {
    'phase_2_lb_super_drawer': 20,
    'phase_2_lb_super_drawer_bsi': 21,
    'phase_2_eb_super_drawer': 22,
    'phase_2_eb_super_drawer_bsi': 23,
    'short_phase_2_eb_super_drawer': 24
}

EB_SUB_DRAWERS = [
    'mini_drawer',
    'mini_drawer',
    'micro_drawer_4',
    'mini_drawer',
    'micro_drawer_5'
]

EB_SHORT_SUB_DRAWERS = [
    'mini_drawer',
    'mini_drawer',
    'micro_drawer_6',
    'mini_drawer',
    'micro_drawer_7'
]

SUB_DRAWERS = {
    'phase_2_lb_super_drawer': 4*['mini_drawer'],
    'phase_2_lb_super_drawer_bsi': 4*['mini_drawer'],
    'phase_2_eb_super_drawer': EB_SUB_DRAWERS,
    'phase_2_eb_super_drawer_bsi': EB_SUB_DRAWERS,
    'short_phase_2_eb_super_drawer': EB_SHORT_SUB_DRAWERS
}

DRAWER_SYSTEM = {
    'phase_2_lb_super_drawer': 'LB',
    'phase_2_lb_super_drawer_bsi': 'LB',
    'phase_2_eb_super_drawer': 'EB',
    'phase_2_eb_super_drawer_bsi': 'EB',
    'short_phase_2_eb_super_drawer': 'EB'
}

In [3]:
def get_sd_id(institute: str, sd_type: str, sd_numbers: range, pre_production: bool):
    df = pd.DataFrame(
        columns=['id', 'type_name', 'location', 'status', 'owner']
    )
    pre_production = int(pre_production)
    sd_ids = []
    sd_types = []
    md_ids = []
    md_types = []
    for sd_number in sd_numbers:
        id_start = \
            f'20{DRAWER_SYSTEM[sd_type]}{MNES[institute]}{SD_TYPES[sd_type]}{sd_number:03d}{pre_production}'
        sd_ids.append(f'{id_start}0')
        sd_types.append(sd_type)
        md_ids.extend([
            f'{id_start}{i}' for i in range(1,len(SUB_DRAWERS[sd_type])+1)
        ])
        md_types.extend(SUB_DRAWERS[sd_type])
    df['id'] = sd_ids + md_ids
    df['type_name'] = sd_types + md_types
    df['location'] = 'B175'
    df['status'] = 'Stored'
    df['owner'] = MNES[institute]
    return df

In [4]:
all_drawers = pd.concat([
    get_sd_id('barcelona', 'phase_2_lb_super_drawer', range(1,3), True),
    get_sd_id('cluj', 'phase_2_lb_super_drawer', range(3,5), True),
    get_sd_id('barcelona', 'phase_2_lb_super_drawer', range(5,58), False),
    get_sd_id('cluj', 'phase_2_lb_super_drawer', range(58,135), False),
    get_sd_id('cluj', 'phase_2_lb_super_drawer_bsi', range(135,137), False),
    get_sd_id('barcelona', 'phase_2_eb_super_drawer', range(137,138), True),
    get_sd_id('cluj', 'phase_2_eb_super_drawer', range(138,141), True),
    get_sd_id('barcelona', 'phase_2_eb_super_drawer', range(141,193), False),
    get_sd_id('cluj', 'phase_2_eb_super_drawer', range(193,263), False),
    get_sd_id('cluj', 'phase_2_eb_super_drawer_bsi', range(263,271), False),
    get_sd_id('cluj', 'short_phase_2_eb_super_drawer', range(271,273), False)
])
all_drawers

,id,type_name,location,status,owner
0,20LBFAE2000110,phase_2_lb_super_drawer,B175,Stored,FAE
1,20LBFAE2000210,phase_2_lb_super_drawer,B175,Stored,FAE
2,20LBFAE2000111,mini_drawer,B175,Stored,FAE
3,20LBFAE2000112,mini_drawer,B175,Stored,FAE
4,20LBFAE2000113,mini_drawer,B175,Stored,FAE
...,...,...,...,...,...
7,20EBCLJ2427201,mini_drawer,B175,Stored,CLJ
8,20EBCLJ2427202,mini_drawer,B175,Stored,CLJ
9,20EBCLJ2427203,micro_drawer_6,B175,Stored,CLJ
10,20EBCLJ2427204,mini_drawer,B175,Stored,CLJ


In [5]:
for type_name, df in all_drawers.groupby('type_name'):
    df.to_csv(f'{type_name}_generated_ids.csv', index=False)